In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

print("Day 16 - RAGAs Evaluation")

Day 16 - RAGAs Evaluation


In [3]:
print("=== RAGAs — 4 Core Metrics ===\n")

# Before we code — understand what each metric measures

metrics_explained = {
    "Faithfulness": {
        "question": "Is every claim in the answer supported by the retrieved context?",
        "good_example": {
            "context": "RAG uses BM25 and vector search merged with RRF.",
            "answer": "RAG uses BM25 and vector search.",
            "score": 1.0,
            "reason": "Every claim in answer exists in context"
        },
        "bad_example": {
            "context": "RAG uses BM25 and vector search merged with RRF.",
            "answer": "RAG uses BM25, vector search and neural reranking.",
            "score": 0.67,
            "reason": "'neural reranking' not in context — hallucination"
        }
    },
    "Answer Relevancy": {
        "question": "Does the answer actually address the question asked?",
        "good_example": {
            "question": "What is BM25?",
            "answer": "BM25 is a keyword search algorithm using TF-IDF.",
            "score": 0.95,
            "reason": "Directly answers what BM25 is"
        },
        "bad_example": {
            "question": "What is BM25?",
            "answer": "Search engines are very important for finding information.",
            "score": 0.2,
            "reason": "Doesn't answer the specific question"
        }
    },
    "Context Recall": {
        "question": "Did the retriever fetch all chunks needed to answer?",
        "good_example": {
            "question": "How does hybrid search work?",
            "retrieved": "BM25 + vector search + RRF chunk",
            "score": 1.0,
            "reason": "All needed information was retrieved"
        },
        "bad_example": {
            "question": "How does hybrid search work?",
            "retrieved": "Only BM25 chunk — RRF chunk missing",
            "score": 0.5,
            "reason": "Half the needed information was missing"
        }
    },
    "Context Precision": {
        "question": "Were the retrieved chunks actually useful?",
        "good_example": {
            "retrieved": "3 chunks all about hybrid search",
            "score": 1.0,
            "reason": "All retrieved chunks were relevant"
        },
        "bad_example": {
            "retrieved": "1 relevant chunk + 2 irrelevant chunks",
            "score": 0.33,
            "reason": "Most retrieved chunks were noise"
        }
    }
}

for metric, info in metrics_explained.items():
    print(f"{'='*50}")
    print(f"Metric: {metric}")
    print(f"Question: {info['question']}")
    print(f"Good score: {info['good_example']['score']} — {info['good_example']['reason']}")
    print(f"Bad score:  {info['bad_example']['score']} — {info['bad_example']['reason']}")
    print()

=== RAGAs — 4 Core Metrics ===

Metric: Faithfulness
Question: Is every claim in the answer supported by the retrieved context?
Good score: 1.0 — Every claim in answer exists in context
Bad score:  0.67 — 'neural reranking' not in context — hallucination

Metric: Answer Relevancy
Question: Does the answer actually address the question asked?
Good score: 0.95 — Directly answers what BM25 is
Bad score:  0.2 — Doesn't answer the specific question

Metric: Context Recall
Question: Did the retriever fetch all chunks needed to answer?
Good score: 1.0 — All needed information was retrieved
Bad score:  0.5 — Half the needed information was missing

Metric: Context Precision
Question: Were the retrieved chunks actually useful?
Good score: 1.0 — All retrieved chunks were relevant
Bad score:  0.33 — Most retrieved chunks were noise



In [4]:
print("=== Building Evaluation Dataset ===\n")

# An evaluation dataset has 4 components per question:
# 1. question     — what was asked
# 2. answer       — what the pipeline returned
# 3. contexts     — what chunks were retrieved
# 4. ground_truth — what the correct answer should be

eval_dataset = [
    {
        "question": "How does hybrid search work?",
        "answer": "Hybrid search combines BM25 keyword search with vector semantic search. Results are merged using Reciprocal Rank Fusion which uses rank position not raw scores.",
        "contexts": [
            "Hybrid search combines BM25 keyword search with vector semantic search. Results are merged using Reciprocal Rank Fusion which uses rank position not raw scores.",
            "BM25 ranks documents based on keyword frequency and inverse document frequency.",
            "Vector search uses embeddings to find semantically similar documents."
        ],
        "ground_truth": "Hybrid search combines BM25 keyword search with vector semantic search, merging results using Reciprocal Rank Fusion."
    },
    {
        "question": "What metrics does RAGAs use?",
        "answer": "RAGAs uses four metrics: faithfulness, answer relevancy, context recall and context precision.",
        "contexts": [
            "RAGAs evaluates RAG pipelines using four metrics: faithfulness, answer relevancy, context recall and context precision.",
            "Faithfulness measures if answers are grounded in retrieved context.",
            "Answer relevancy measures if the answer addresses the question asked."
        ],
        "ground_truth": "RAGAs uses faithfulness, answer relevancy, context recall and context precision."
    },
    {
        "question": "How is multi-tenancy implemented?",
        "answer": "Multi-tenancy is implemented through ChromaDB collection isolation where each organization gets a separate collection.",
        "contexts": [
            "Multi-tenancy is implemented through ChromaDB collection isolation. Each organization gets a separate collection.",
            "Users cannot access other organizations data due to collection level isolation.",
            "JWT tokens control user authentication and document access."
        ],
        "ground_truth": "Multi-tenancy uses ChromaDB collection isolation giving each organization a separate collection."
    },
    {
        "question": "What does LoRA do?",
        "answer": "LoRA reduces trainable parameters by 90 percent using low-rank matrix decomposition and adds adapter matrices.",
        "contexts": [
            "LoRA fine-tuning reduces trainable parameters by 90 percent using low-rank matrix decomposition.",
            "It adds small adapter matrices to existing weights instead of modifying them directly.",
            "This allows fine-tuning large models on consumer hardware."
        ],
        "ground_truth": "LoRA reduces trainable parameters by 90 percent using low-rank matrix decomposition."
    },
    {
        "question": "How is the pipeline deployed?",
        "answer": "The pipeline is deployed on AWS EC2 with Nginx as reverse proxy and GitHub Actions for CI/CD.",
        "contexts": [
            "The pipeline is deployed on AWS EC2 with Nginx as reverse proxy.",
            "GitHub Actions handles CI/CD — auto deploys on push to main branch.",
            "Docker containerization ensures consistent environments across deployments."
        ],
        "ground_truth": "The pipeline runs on AWS EC2 with Nginx reverse proxy and GitHub Actions CI/CD."
    },
    {
        "question": "What is the pricing for enterprise?",
        "answer": "I cannot find this information in the provided documents.",
        "contexts": [
            "Hybrid search combines BM25 and vector search.",
            "RAGAs evaluates faithfulness and answer relevancy.",
            "FastAPI handles async request routing."
        ],
        "ground_truth": "The pricing information is not available in the documents."
    }
]

print(f"Evaluation dataset: {len(eval_dataset)} questions")
print("\nQuestions:")
for i, item in enumerate(eval_dataset):
    print(f"  {i+1}. {item['question']}")

=== Building Evaluation Dataset ===

Evaluation dataset: 6 questions

Questions:
  1. How does hybrid search work?
  2. What metrics does RAGAs use?
  3. How is multi-tenancy implemented?
  4. What does LoRA do?
  5. How is the pipeline deployed?
  6. What is the pricing for enterprise?


In [7]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import Dataset

print("=== Running RAGAs Evaluation ===\n")

# RAGAs needs an LLM to evaluate — we use Groq
eval_llm = LangchainLLMWrapper(
    ChatGroq(
        api_key=os.getenv("GROQ_API_KEY"),
        model="llama-3.1-8b-instant",
        temperature=0
    )
)

# RAGAs needs embeddings for answer relevancy
eval_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

# Convert to HuggingFace Dataset format
hf_dataset = Dataset.from_list(eval_dataset)
print(f"Dataset converted: {hf_dataset}")

# Run evaluation
print("\nRunning evaluation — this takes 2-3 minutes...\n")

results = evaluate(
    dataset=hf_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_recall,
        context_precision
    ],
    llm=eval_llm,
    embeddings=eval_embeddings
)

print("\n=== RAGAs Scores ===")
print(results)

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [5]:
from groq import Groq
import json

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def compute_faithfulness(
    answer: str,
    contexts: list,
    client: Groq
) -> float:
    """
    Faithfulness measures if every claim in the answer
    is supported by the retrieved context.
    
    Method:
    1. Extract claims from the answer
    2. For each claim check if context supports it
    3. Score = supported claims / total claims
    """
    context_str = "\n".join(contexts)
    
    # Step 1 — extract claims
    claims_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{
            "role": "user",
            "content": f"""Extract all factual claims from this answer as a JSON list.
Return ONLY a JSON array of strings, no other text.

Answer: {answer}

Example output: ["claim 1", "claim 2", "claim 3"]"""
        }],
        temperature=0
    )
    
    try:
        claims_text = claims_response.choices[0].message.content.strip()
        # Clean markdown if present
        claims_text = claims_text.replace("```json", "").replace("```", "").strip()
        claims = json.loads(claims_text)
    except:
        claims = [answer]
    
    if not claims:
        return 1.0
    
    # Step 2 — verify each claim against context
    supported = 0
    claim_results = []
    
    for claim in claims:
        verify_response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{
                "role": "user",
                "content": f"""Does the context support this claim?
Answer with only YES or NO.

Context: {context_str}

Claim: {claim}"""
            }],
            temperature=0,
            max_tokens=5
        )
        verdict = verify_response.choices[0].message.content.strip().upper()
        is_supported = "YES" in verdict
        if is_supported:
            supported += 1
        claim_results.append({
            "claim": claim,
            "supported": is_supported
        })
    
    score = supported / len(claims)
    return score, claim_results


# Test faithfulness
print("=== Faithfulness Metric ===\n")

# Good answer — grounded in context
good_answer = "Hybrid search combines BM25 and vector search merged with RRF."
good_contexts = [
    "Hybrid search combines BM25 keyword search with vector semantic search.",
    "Results are merged using Reciprocal Rank Fusion (RRF)."
]

score, claims = compute_faithfulness(good_answer, good_contexts, groq_client)
print(f"Good answer faithfulness: {score:.2f}")
print(f"Claims checked:")
for c in claims:
    status = "✅" if c['supported'] else "❌"
    print(f"  {status} {c['claim']}")

# Bad answer — hallucinated claim
print()
bad_answer = "Hybrid search combines BM25 and vector search. It also uses neural reranking and GPT-4 for generation."
bad_contexts = [
    "Hybrid search combines BM25 keyword search with vector semantic search.",
    "Results are merged using Reciprocal Rank Fusion (RRF)."
]

score2, claims2 = compute_faithfulness(bad_answer, bad_contexts, groq_client)
print(f"Bad answer faithfulness: {score2:.2f}")
print(f"Claims checked:")
for c in claims2:
    status = "✅" if c['supported'] else "❌"
    print(f"  {status} {c['claim']}")

=== Faithfulness Metric ===

Good answer faithfulness: 0.80
Claims checked:
  ✅ Hybrid search combines BM25
  ❌ BM25 and vector search are merged
  ✅ vector search is a thing
  ✅ RRF is a thing
  ✅ BM25 and vector search are merged with RRF

Bad answer faithfulness: 0.33
Claims checked:
  ✅ Hybrid search combines BM25 and vector search.
  ❌ Hybrid search also uses neural reranking.
  ❌ Hybrid search also uses GPT-4 for generation.


In [ ]:
def compute_answer_relevancy(
        question: str,
        answer: str,
        client: Groq,
        n_questions: int=3
) -> float:
    """ 
    Answer relevancy measures if the answer addresses the question.

    Method:
    1. Generate n questions that the answer could be answering
    2. Compute similarity between original question and generated questions.
    3. Higher similarity = answer is more relevant to the question
    """
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np

    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    # Generate questions from the answer
    gen_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        message=[{
            "role": "user",
            "content": f"""Generate {n_questions} different questions that this answer is responding to.
Return ONLY a JSON array of strings, no other texts.

Answer: {answer}

Example output: ["question 1?", "question 2?", "question 3?"]"""
        }],
        temprature=0.3 
    )

    try: 
        gen_text = gen_response.choices[0].message.content.strip()
        gen_text  = gen_text.replace("```json", "").replace("```", "").strip()
        generated_questions = json.loads(gen_text)
    except:
        generated_quesions = [question]

    # Compute similarity between original and generated questions
    original_emb = embedder.encode([question])
    generated_embs = embedder.encode(generated_questions)
    similarities = cosine_similarity(original_emb, generated_embs)[0]
    score = float(np.mean(similarities))

    return score, generated_questions

print("=== Answer Relevancy Metric ===\n")

# Relevant answer
q1 = "What is BM25?"
a1 = "BM25 is a keyword search algorithm that ranks documents using tern frequency and inverse document frequency."
score1, gen_q1 = compute_answer_relevancy(q1, a1, groq_client)
print(f"Question: {q1}")
print(f"Answer: {a1[:60]}...")
print(f"Generated questions: {gen_q1}")
print(f"Relevancy score: {score:.4f}\n")

# Irrelevant answer
q2 = "What is BM25?"
a2 = "Search engines are very important tools for finding information on the internet and helping users navigate content."
score2, gen_q2 = compute_answer_relevancy(q2, a2, groq_client)
print(f"Question: {q2}")
print(f"Answer: {a2[:60]}...")
print(f"Generated questions: {gen_q2}")
print(f"Relevancy score: {score2:.4f}")

=== Answer Relevancy Metric ===

